In [1]:
import pandas as pd, numpy as np

s = pd.read_parquet("data/clean/scores.parquet")
val = s[s["period"] == "valid"].copy()
days_val = val["ts"].dt.date.nunique()

# --- £ assumptions (illustrative, edit freely) ---
REVIEW_COST = 3.0            # analyst time per alert (~5 min of a £36/hour analyst)
FALSE_DECLINE_COST = 10.0    # genuine customer wrongly declined: lost sale + annoyance + complaint risk
STEPUP_FRICTION = 0.50       # genuine customer asked for a one-time passcode
STEPUP_STOPS_FRAUD = 0.80    # share of fraud attempts a one-time passcode stops

print(f"Validation: {len(val):,} transactions over {days_val} days | fraud: {val['is_fraud'].sum():,} "
      f"| fraud value £{val.loc[val['is_fraud']==1, 'amt'].sum():,.0f}")

Validation: 401,883 transactions over 181 days | fraud: 2,419 | fraud value £1,293,541


In [4]:
def cost_of_policy(d, t_step, t_decline):
    fraud, amt, sc = d["is_fraud"].values == 1, d["amt"].values, d["score"].values
    decline = sc >= t_decline
    step = (sc >= t_step) & ~decline
    approve = ~decline & ~step
    fraud_loss = amt[fraud & approve].sum() + (1 - STEPUP_STOPS_FRAUD) * amt[fraud & step].sum()
    friction = FALSE_DECLINE_COST * (decline & ~fraud).sum() + STEPUP_FRICTION * (step & ~fraud).sum()
    review = REVIEW_COST * decline.sum()
    n_days = d["ts"].dt.date.nunique()
    return {"t_step": t_step, "t_decline": t_decline,
            "fraud_loss": fraud_loss, "friction_cost": friction, "review_cost": review,
            "total_cost": fraud_loss + friction + review,
            "alerts_per_day": decline.sum() / n_days,
            "stepups_per_day": step.sum() / n_days,
            "fraud_value_stopped_%": 100 * (1 - fraud_loss / amt[fraud].sum()),
            "false_declines_per_10k_genuine": 10_000 * (decline & ~fraud).sum() / (~fraud).sum()}

grid = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]
res = pd.DataFrame([cost_of_policy(val, a, b) for b in grid for a in grid if a <= b])

nothing = cost_of_policy(val, 2, 2)    # cut-offs above 1 = approve everything
decline_only = res[res["t_step"] == res["t_decline"]].nsmallest(1, "total_cost").iloc[0]
with_stepup = res.nsmallest(1, "total_cost").iloc[0]

summary = pd.DataFrame([nothing, decline_only, with_stepup],
                       index=["No model (approve all)", "Model: decline only", "Model: step-up + decline"])
summary["total_cost_per_month"] = summary["total_cost"] / days_val * 30
res.to_csv("outputs/policy_grid.csv", index=False)
summary.round(2).T

,No model (approve all),Model: decline only,Model: step-up + decline
t_step,2.00,0.01,0.00
t_decline,2.00,0.01,0.10
fraud_loss,1293540.67,51132.89,46695.65
friction_cost,0.00,12420.00,5907.50
review_cost,0.00,10350.00,7851.00
total_cost,1293540.67,73902.89,60454.15
alerts_per_day,0.00,19.06,14.46
stepups_per_day,0.00,0.00,14.49
fraud_value_stopped_%,0.00,96.05,96.39
false_declines_per_10k_genuine,0.00,31.09,11.62


In [7]:
sens = []
for fdc in [5, 10, 20, 40]:
    FALSE_DECLINE_COST = fdc
    r = pd.DataFrame([cost_of_policy(val, a, b) for b in grid for a in grid if a <= b]).nsmallest(1, "total_cost").iloc[0]
    sens.append({"false_decline_cost_£": fdc, "best_t_step": r["t_step"], "best_t_decline": r["t_decline"],
                 "alerts_per_day": round(r["alerts_per_day"], 1),
                 "stepups_per_day": round(r["stepups_per_day"], 1),
                 "false_declines_per_10k": round(r["false_declines_per_10k_genuine"], 1),
                 "fraud_value_stopped_%": round(r["fraud_value_stopped_%"], 1)})
FALSE_DECLINE_COST = 10.0
pd.DataFrame(sens)

,false_decline_cost_£,best_t_step,best_t_decline,alerts_per_day,stepups_per_day,false_declines_per_10k,fraud_value_stopped_%
0,5,0.005,0.10,14.5,14.5,11.6,96.4
1,10,0.005,0.10,14.5,14.5,11.6,96.4
2,20,0.005,0.15,14.0,15.0,9.8,96.3
3,40,0.005,0.15,14.0,15.0,9.8,96.3


In [8]:
import json
best = res.nsmallest(1, "total_cost").iloc[0]
policy = {"t_step": float(best["t_step"]), "t_decline": float(best["t_decline"]),
          "REVIEW_COST": REVIEW_COST, "FALSE_DECLINE_COST": FALSE_DECLINE_COST,
          "STEPUP_FRICTION": STEPUP_FRICTION, "STEPUP_STOPS_FRAUD": STEPUP_STOPS_FRAUD}
json.dump(policy, open("outputs/best_policy.json", "w"), indent=2)
policy

{'t_step': 0.005,
 't_decline': 0.1,
 'REVIEW_COST': 3.0,
 'FALSE_DECLINE_COST': 10.0,
 'STEPUP_FRICTION': 0.5,
 'STEPUP_STOPS_FRAUD': 0.8}